# Imports 

In [2]:
import pandas as pd
from collections import defaultdict
import sys
import os
import shutil as sh
import urllib
import tarfile
import numpy as np
import math
import seaborn as sns
import glob, os
import importlib
import gzip
import MDAnalysis as mda
import nglview as nv
import requests
import json
from biopandas.pdb import PandasPdb
from Bio import AlignIO
import re

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.spatial import ConvexHull

from urllib.error import HTTPError
from pathlib import Path
from ipywidgets import interact, interactive, fixed, interact_manual, IntProgress
import ipywidgets as widgets # type: ignore
from IPython.display import display, Markdown, clear_output

#Pandarallel works only on linux and mac
try:
    from pandarallel import pandarallel
    pandarallel.initialize(nb_workers=8,progress_bar=True)
    PARRALEL = True
except:
    PARRALEL = False

from tqdm.notebook import tnrange, tqdm
tqdm.pandas() #activate tqdm progressbar for pandas apply

#Pandas configuration
pd.options.mode.chained_assignment = (
    None  # default='warn', remove pandas warning when adding a new column
)

pd.set_option("display.max_columns", None)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
%config InlineBackend.figure_format ='svg' #better quality figure figure

#%matplotlib inline
sns.set_style("darkgrid")

np.seterr(divide='ignore', invalid='ignore')



INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

# Dataset Generation 

In [ ]:
%run "./data_preparation.ipynb"
UPDATE = False

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
         Domain  Class  Architecture  Topology  Homologous  S35  S60  S95  \
0       1oaiA00      1            10         8          10    1    1    1   
1       1go5A00      1            10         8          10    1    1    1   
2       3frhA01      1            10         8          10    2    1    1   
3       3friA01      1            10         8          10    2    1    1   
4       3b89A01      1            10         8          10    2    1    1   
...         ...    ...           ...       ...         ...  ...  ...  ...   
434852  2kn1A00      4            10      1290          10    2    1    1   
434853  1vprA01      4            10      1300          10    1    1    1   
434854  1vprA02      4            10      1310          10    1    1    1   
434855  1jyoE00      4            10      1330          10    1    1    1   
434856  1jy

  0%|          | 0/434857 [00:00<?, ?it/s]

In [1]:
RECALCULATION = True 

In [3]:
recalculation_widget = widgets.ToggleButton(
    value=RECALCULATION,
    description='Recalculation ?',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click for recalculation',
    icon='cogs' # (FontAwesome names without the `fa-` prefix)
)
display(recalculation_widget)

ToggleButton(value=True, description='Recalculation ?', icon='cogs', tooltip='Click for recalculation')

In [ ]:
import pepr2ds.builder.Builder as builderEngine
importlib.reload(builderEngine)
builder = builderEngine.Builder(SETUP, recalculate = recalculation_widget.value, update=False, notebook = True, core=1)

In [ ]:
builder.structure.clean_all_pdbs()

In [ ]:
# Verify that the **builder.structure.clean_all_pdbs()** worked 

raw = glob.glob(os.path.join(SETUP["CATHFOLDER"], "domains", "*", "raw", "*.pdb"))
cleaned = glob.glob(os.path.join(SETUP["CATHFOLDER"], "domains", "*", "cleaned", "*.pdb"))

print(f"Found {len(raw)} raw PDBs and {len(cleaned)} cleaned PDBs.")

# it should display same number or raw PDBs and cleaned ones

In [ ]:
# Check if dssp works using just one cleaned pdb
# before doing this for every cleaned pdb file 
# insert header to one cleaned to check if the error was fixed 
import os
# Point Biopython at the mkdssp binary via the DSSP env var
os.environ["DSSP"] = "/usr/local/bin/mkdssp"

# Now import
from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP

# insert header 
from pathlib import Path

path = Path("/home/user_stel/AISB/Project/databases/cath/domains/ANNEXIN/cleaned/1a8aA01.pdb")
lines = path.read_text().splitlines(keepends=True)

# Insert HEADER if missing
if not lines[0].startswith("HEADER"):
    lines.insert(0, "HEADER    GENERATED BY PePrMInt CLEANER\n")
    path.write_text("".join(lines))
    print("Inserted HEADER.")

# Verify
print("First lines now:")
print("".join(lines[:10]))

# Quick smoke test on one PDB
parser = PDBParser(QUIET=True)
structure = parser.get_structure("test", "/home/user_stel/AISB/Project/databases/cath/domains/ANNEXIN/cleaned/1a8aA01.pdb")
dssp = DSSP(structure[0], "/home/user_stel/AISB/Project/databases/cath/domains/ANNEXIN/cleaned/1a8aA01.pdb", dssp="mkdssp")
print("DSSP ran, first few residues:", list(dssp)[:5])

In [ ]:
# Insert header to every cleaned pdb 
from pathlib import Path

cleaned_dir = Path("/home/user_stel/AISB/Project/databases/cath/domains")
for pdb_path in cleaned_dir.glob("**/cleaned/*.pdb"):
    lines = pdb_path.read_text().splitlines(keepends=True)
    # If the first non-blank line isn't a HEADER, insert one
    if not lines or not lines[0].startswith("HEADER"):
        header = "HEADER    PePrMInt cleaned PDB\n"
        
        lines.insert(0, header)
        pdb_path.write_text("".join(lines))
        print(f"Inserted HEADER into {pdb_path}")

In [ ]:
base = Path("/home/user_stel/AISB/Project/databases/cath/domains")

# 3) Loop over each domain subfolder
for domain_dir in base.iterdir():
    raw_dir = domain_dir / "raw"
    if not raw_dir.is_dir():
        continue

    # 4) Clean every PDB in that raw/ folder
    for pdb_path in raw_dir.glob("*.pdb"):
        cleaned_path = builder.structure.clean_pdb(str(pdb_path))
        print(f"Cleaned {pdb_path}")

In [ ]:
# test if all the hydrogen atoms are indeed removed 
import re

with open("/home/user_stel/AISB/Project/databases/cath/domains/ANNEXIN/cleaned/1a8aA01.pdb") as f:
    hydrogens = [L for L in f
                 if (L.startswith(("ATOM","HETATM"))
                     and re.match(r'^[\sA-Za-z0-9]*H', L[12:16]))]
print("Leftover H-atoms:", hydrogens)


In [ ]:
# For some reason the dssp did not see any header in the cleaned pdbs
# in order to make sure that there was no headr we ran in the terminal the following command:
# head -n 5 /home/user_stel/AISB/Project/databases/cath/domains/ANNEXIN/cleaned/1a8aA01.pdb
# which confirmed that the cleaned pdbs in fact did not have headers 
# so we inserted headers to every cleaned pdb
df_struc = builder.structure.build_structural_dataset()